# HyperKT Stage 4 — Forgetting-aware causal Transformer
**Runtime:** Google Colab **T4 GPU** (Python 3). Upload `hyperkt_stage2.zip` and `hyperkt_stage3b.zip`. This notebook trains **two controlled variants**: temporal-only and frozen-hypergraph fusion. It evaluates **validation** only; the test set remains untouched. This is a low-memory research implementation (d=256, heads=8, layers=4), not a reproduction of paper performance claims. Stage 5 will add Bayesian heads.

In [ ]:
import os,io,json,zipfile,random,copy,shutil
import numpy as np, torch
from torch import nn
from torch.utils.data import DataLoader,TensorDataset
from sklearn.metrics import roc_auc_score, log_loss
from google.colab import files
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',DEVICE,torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU')
assert DEVICE.type=='cuda','Select Runtime > Change runtime type > T4 GPU'


In [ ]:
uploaded=files.upload()
assert 'hyperkt_stage2.zip' in uploaded and 'hyperkt_stage3b.zip' in uploaded,'Upload both requested ZIP files'
z2=zipfile.ZipFile(io.BytesIO(uploaded['hyperkt_stage2.zip']))
z3=zipfile.ZipFile(io.BytesIO(uploaded['hyperkt_stage3b.zip']))
def load_npz(z,name):
    with np.load(io.BytesIO(z.read(name)),allow_pickle=False) as data:
        return {k:data[k].copy() for k in data.files}
train=load_npz(z2,'train_sequences.npz'); val=load_npz(z2,'val_sequences.npz')
hg=load_npz(z3,'hypergraph_repaired.npz')
manifest=json.loads(z2.read('stage2_report.json'))
print('Train:',train['mask'].sum(),'Val:',val['mask'].sum(),'Hypergraph:',hg['assignments'].shape)
assert set(train['user_id']).isdisjoint(set(val['user_id']))
assert np.array_equal(hg['skill_ids'],np.arange(2,151))
assert hg['assignments'].shape==(149,32)


## Leakage checks
Only previous interaction correctness is fed as input. The target skill is known at prediction time, but the target correctness is never embedded. Padding tokens are excluded from both attention and loss. The provided windows are assumed chronological as established in Stage 2; verify original event timestamps separately for publication.

In [ ]:
PAD=1; N_SKILLS=151; MAXLEN=200
for split,d in [('train',train),('val',val)]:
    assert d['history_skill'].shape==d['target_skill'].shape==d['mask'].shape
    assert d['history_skill'].shape[1]==MAXLEN
    assert d['target_correct'][d['mask']].min()>=0 and d['target_correct'][d['mask']].max()<=1
    assert np.all((d['target_skill'][d['mask']]>=2)&(d['target_skill'][d['mask']]<N_SKILLS))
    assert not np.any(d['history_skill'][d['mask']]==PAD)
    print(split,'windows',len(d['mask']),'targets',int(d['mask'].sum()))
def dataset(d):
    names=['history_skill','history_correct','target_skill','target_correct','mask']
    return TensorDataset(*(torch.tensor(d[k],dtype=torch.bool if k=='mask' else torch.long) for k in names))
train_ds=dataset(train);val_ds=dataset(val)


In [ ]:
# Frozen hypergraph embeddings: assignments -> 32-dimensional structural feature per skill
A=np.zeros((N_SKILLS,32),dtype=np.float32)
A[hg['skill_ids']]=hg['assignments'].astype(np.float32)
A[PAD]=0
A=torch.tensor(A)
class HyperKTTemporal(nn.Module):
    def __init__(self, use_hypergraph=False,d=256,heads=8,layers=4,dropout=.1):
        super().__init__();self.use_hypergraph=use_hypergraph
        self.skill=nn.Embedding(N_SKILLS,d,padding_idx=PAD)
        self.response=nn.Embedding(2,d)
        self.pos=nn.Embedding(MAXLEN,d)
        self.decay_raw=nn.Parameter(torch.tensor(-2.97)) # softplus ~0.05
        if use_hypergraph:
            self.register_buffer('assignments',A.clone())
            self.hg_proj=nn.Linear(32,d,bias=False)
            self.hg_gate=nn.Linear(2*d,d)
        enc=nn.TransformerEncoderLayer(d_model=d,nhead=heads,dim_feedforward=4*d,dropout=dropout,batch_first=True,activation='gelu',norm_first=True)
        self.encoder=nn.TransformerEncoder(enc,num_layers=layers,enable_nested_tensor=False)
        self.norm=nn.LayerNorm(d)
        self.head=nn.Sequential(nn.Linear(2*d,d),nn.GELU(),nn.Dropout(dropout),nn.Linear(d,1))
    def forward(self,hs,hc,ts,mask):
        B,L=hs.shape
        pos=torch.arange(L,device=hs.device)
        # Recent positions are given stronger positional weighting. No real-time gaps exist in Stage 2.
        age=(L-1-pos).float()
        decay=torch.exp(-torch.nn.functional.softplus(self.decay_raw)*age/MAXLEN)
        x=self.skill(hs)+self.response(hc.clamp(0,1))+self.pos(pos)[None,:,:]*decay[None,:,None]
        if self.use_hypergraph:
            structure=self.hg_proj(self.assignments[hs]);gate=torch.sigmoid(self.hg_gate(torch.cat([x,structure],dim=-1)))
            x=x+gate*structure
        # Prevent looking ahead at later interaction outcomes
        causal=torch.triu(torch.ones(L,L,device=hs.device,dtype=torch.bool),diagonal=1)
        # Stage 2 may pad the tail; zero padded states and rely on causal masking.
        x=x.masked_fill(~mask.unsqueeze(-1),0.0)
        h=self.norm(self.encoder(x,mask=causal))
        target=self.skill(ts)
        if self.use_hypergraph:target=target+self.hg_proj(self.assignments[ts])
        return self.head(torch.cat([h,target],dim=-1)).squeeze(-1)


## Training and validation
Selection uses validation AUC, not test performance. GPU memory usage may require lowering `BATCH_SIZE` to 8. Keep the same seed and settings for both variants.

In [ ]:
from torch.nn import functional as F
BATCH_SIZE=16;EPOCHS=20;PATIENCE=5;LR=2e-4
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=True)
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
def run_epoch(model,loader,optimizer=None):
    training=optimizer is not None;model.train(training)
    losses=[];ys=[];ps=[]
    for hs,hc,ts,y,m in loader:
        hs,hc,ts,y,m=[v.to(DEVICE,non_blocking=True) for v in (hs,hc,ts,y,m)]
        if training:optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits=model(hs,hc,ts,m)
            loss=F.binary_cross_entropy_with_logits(logits[m],y[m].float())
            if training:loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.0);optimizer.step()
        losses.append((float(loss.detach())*int(m.sum()),int(m.sum())))
        if not training:
            ys.append(y[m].detach().cpu().numpy());ps.append(torch.sigmoid(logits[m]).detach().cpu().numpy())
    out={'nll':sum(v for v,c in losses)/sum(c for v,c in losses)}
    if not training:
        y=np.concatenate(ys);p=np.concatenate(ps)
        out['auc']=float(roc_auc_score(y,p));out['n_targets']=int(len(y))
    return out


In [ ]:
os.makedirs('/content/hyperkt_stage4',exist_ok=True)
all_reports={}
for use_hg,name in [(False,'temporal_only'),(True,'hypergraph_fusion')]:
    torch.manual_seed(SEED)
    if DEVICE.type=='cuda':torch.cuda.manual_seed_all(SEED)
    model=HyperKTTemporal(use_hypergraph=use_hg).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=0.01)
    best_auc=-1;best_state=None;best_epoch=0;wait=0;history=[]
    for epoch in range(1,EPOCHS+1):
        tr=run_epoch(model,train_loader,opt);va=run_epoch(model,val_loader)
        history.append({'epoch':epoch,'train_nll':tr['nll'],'val_nll':va['nll'],'val_auc':va['auc']})
        print(name,epoch,'train NLL',round(tr['nll'],4),'val AUC',round(va['auc'],5),flush=True)
        if va['auc']>best_auc+1e-5:
            best_auc=va['auc'];best_epoch=epoch;best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};wait=0
        else:
            wait+=1
            if wait>=PATIENCE:break
    torch.save({'state_dict':best_state,'model':name,'seed':SEED,'best_epoch':best_epoch,'config':{'d':256,'heads':8,'layers':4,'max_len':MAXLEN}},f'/content/hyperkt_stage4/{name}.pt')
    import pandas as pd
    pd.DataFrame(history).to_csv(f'/content/hyperkt_stage4/{name}_history.csv',index=False)
    all_reports[name]={'best_val_auc':best_auc,'best_epoch':best_epoch,'history':history}
    del model,opt;torch.cuda.empty_cache()


In [ ]:
with open('/content/hyperkt_stage4/report.json','w') as f:json.dump(all_reports,f,indent=2)
with open('/content/hyperkt_stage4/protocol.txt','w') as f:
    f.write('Stage 4: student-disjoint train/validation; causal attention; validation-only model selection; test unopened. Hypergraph assignments frozen from Stage 3B. Position-based decay, NOT timestamp-based forgetting.\n')
shutil.make_archive('/content/hyperkt_stage4','zip','/content/hyperkt_stage4')
print('Saved /content/hyperkt_stage4.zip')
files.download('/content/hyperkt_stage4.zip')
